# Peering Inside the Overton Window: A Computational Analysis of Discourse Shift in US News Headlines (2001–2024)

**SI 630: Natural Language Processing · Winter 2026**  
**Paris Heard · School of Information, University of Michigan**

---

This notebook contains the **complete analysis pipeline** for the paper *Peering Inside the Overton Window* (Heard, 2026). It is organized to mirror the paper's structure and is intended to be fully reproducible from top to bottom.

### Research Question
Can the drift of the Overton Window — the range of political ideas considered acceptable in mainstream discourse — be measured and characterized using NLP methods applied to large-scale news headline corpora?

### Pipeline Overview
| Section | Description |
|---|---|
| 0 | Imports & configuration |
| 1 | Data loading (HuggingFace, restricted access) |
| 2 | Preprocessing & corpus split |
| 3 | TF-IDF vectorization |
| 4 | Monthly cosine drift (§5.1) |
| 5 | Permutation test (§5.2) |
| 6 | Topic-controlled analysis (§5.3) |
| 7 | Sentence embeddings (§5.4) |
| 8 | UMAP visualization (§5.5) |
| 9 | Sentiment analysis (§5.6) |
| 10 | Save results |

### Data Access
This notebook requires authenticated HuggingFace access to `dess-mannheim/US_Multi_Outlet_News_Headlines2001_2024`. Request access at the dataset page, then run `huggingface-cli login` or the login cell below before executing.

## 0. Imports & Configuration

In [ ]:
# HuggingFace authentication — required before dataset loading
# Run this cell first, then proceed with the rest of the notebook
from huggingface_hub import login
login()

In [ ]:
import re
import sys
from pathlib import Path
from tqdm import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats

# Import reusable functions from the pipeline module
sys.path.insert(0, str(Path('..').resolve()))
from src.overton_pipeline import (
    parse_date, preprocess, is_valid,
    build_tfidf, cosine_distance,
    compute_monthly_drift, permutation_test,
    compute_sentiment, sentiment_conditioned_drift,
    run_filtered_analysis, top_tfidf_terms,
    plot_drift, plot_permutation_test,
    plot_sentiment, plot_sentiment_conditioned_drift,
    plot_topic_filters, save_results,
    TOPIC_FILTERS, ARTIFACT_TERMS,
)

# ── Corpus parameters (§2, §4.2) ──────────────────────────────────────────────
BASELINE_START,   BASELINE_END   = 2001, 2015
COMPARISON_START, COMPARISON_END = 2016, 2024
ALL_OUTLETS = ['nyt', 'lat', 'wsj', 'foxnews', 'msnbc']

# ── TF-IDF parameters (§4.3) ──────────────────────────────────────────────────
MAX_FEATURES = 10_000
NGRAM_RANGE  = (1, 2)   # unigrams + bigrams to capture compound political terms
MIN_DF       = 5        # reduce noise from rare terms
# sublinear_tf=True and stop_words='english' applied in build_tfidf()

# ── Output paths ──────────────────────────────────────────────────────────────
FIG_DIR = Path('outputs/figures')
RES_DIR = Path('outputs/results')
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

## 1. Data Loading

The primary dataset is `dess-mannheim/US_Multi_Outlet_News_Headlines2001_2024` (Chair for Data Science in the Economic and Social Sciences, University of Mannheim, 2026), streamed from HuggingFace. It contains 6,235,881 raw headlines from five US outlets spanning January 2001 – December 2024.

**Access:** Restricted to research use. Authenticate with HuggingFace before running this cell (see cell 0 above).

In [ ]:
from datasets import load_dataset

ds = load_dataset('dess-mannheim/US_Multi_Outlet_News_Headlines2001_2024', 'raw')
print(ds)

# Quick sanity check on schema
print('\nColumn names:', ds['nyt'].column_names)
print('Sample record:', ds['nyt'][0])

In [ ]:
# Outlet-level overview
print(f'{'Outlet':<12} {'Total Headlines':>18}')
print('-' * 32)
total = 0
for outlet in ALL_OUTLETS:
    n = ds[outlet].num_rows
    total += n
    print(f'{outlet:<12} {n:>18,}')
print(f'{'TOTAL':<12} {total:>18,}')

## 2. Preprocessing & Corpus Split

Headlines are preprocessed using the pipeline described in §4.1 of the paper: lowercase, URL removal, non-alphanumeric stripping (retaining hyphens for compound terms), minimum 4-word filter. Date fields are normalized to YYYY-MM monthly bins (§4.2). The corpus is then divided into:

- **Baseline** (2001–2015): 180 monthly bins — a stable reference period of US political discourse
- **Comparison** (2016–2024): 108 monthly bins — period of documented political disruption

Note on outlet composition: Fox News data begins in 2014, so it appears sparsely in the baseline. This compositional shift (Fox News: 20.6% of baseline → 39.5% of comparison) is discussed as a limitation in §6.5.

In [ ]:
baseline_records, comparison_records = [], []

for outlet in ALL_OUTLETS:
    print(f'Processing {outlet}...', end='', flush=True)
    b_count = c_count = 0

    for row in ds[outlet]:
        date = parse_date(row['date'])
        if date is None:
            continue
        year = int(date[:4])

        headline = preprocess(row['headline'])
        if not is_valid(headline):
            continue

        record = {'title': headline, 'source': outlet, 'year_month': date}

        if BASELINE_START <= year <= BASELINE_END:
            baseline_records.append(record)
            b_count += 1
        elif COMPARISON_START <= year <= COMPARISON_END:
            comparison_records.append(record)
            c_count += 1

    print(f' baseline={b_count:,}  comparison={c_count:,}')

df_baseline   = pd.DataFrame(baseline_records)
df_comparison = pd.DataFrame(comparison_records)
df_baseline['title_clean']   = df_baseline['title']
df_comparison['title_clean'] = df_comparison['title']

# Corpus statistics
total_raw      = sum(ds[o].num_rows for o in ALL_OUTLETS)
total_filtered = len(df_baseline) + len(df_comparison)
print(f'\nRaw total    : {total_raw:,}')
print(f'After filter : {total_filtered:,}  ({(total_raw - total_filtered)/total_raw*100:.1f}% removed)')
print(f'\nBaseline   : {len(df_baseline):,} headlines  ({df_baseline["year_month"].nunique()} monthly bins)')
print(f'Comparison : {len(df_comparison):,} headlines  ({df_comparison["year_month"].nunique()} monthly bins)')

In [ ]:
# Outlet distribution per corpus (Table 1 in paper)
print('=== Outlet Distribution ===')
for corpus_label, df in [('Baseline', df_baseline), ('Comparison', df_comparison)]:
    print(f'\n{corpus_label}:')
    counts = df['source'].value_counts()
    for outlet in ALL_OUTLETS:
        n   = counts.get(outlet, 0)
        pct = n / len(df) * 100
        print(f'  {outlet:<10}: {n:>10,}  ({pct:.1f}%)')

## 3. TF-IDF Vectorization

TF-IDF representations are computed as described in §4.3. The vectorizer is **fit exclusively on the baseline corpus** to fix the feature space — comparison period headlines are projected into the 2001–2015 vocabulary rather than a shifting one. This is a deliberate methodological choice: new vocabulary entering the comparison period (e.g., *covid*, *maga*, *omicron*) falls outside the baseline vocabulary, naturally down-weighting purely topical drift and amplifying structural change (§6.3).

In [ ]:
vectorizer, X_baseline, X_comparison, feature_names, centroid_baseline = build_tfidf(
    df_baseline, df_comparison,
    max_features = MAX_FEATURES,
    ngram_range  = NGRAM_RANGE,
    min_df       = MIN_DF,
)

print('\n=== Top TF-IDF Terms — Baseline ===')
for term, weight in top_tfidf_terms(centroid_baseline, feature_names):
    print(f'  {term:<25} {weight:.4f}')

comp_centroid = np.asarray(X_comparison.mean(axis=0)).flatten()
print('\n=== Top TF-IDF Terms — Comparison ===')
for term, weight in top_tfidf_terms(comp_centroid, feature_names):
    print(f'  {term:<25} {weight:.4f}')

## 4. Monthly Cosine Drift — TF-IDF (§5.1)

Two metrics are computed per monthly bin (§4.5):
- **Cosine distance from baseline centroid** — measures shift in the center of acceptable discourse
- **Intra-month spread** — measures the width of discourse boundaries (std of per-headline cosine similarities to monthly centroid)

The September 2001 spike serves as informal validation: it coincides with the 9/11 attacks and confirms the metric is sensitive to genuine topical events. Importantly, the baseline returns to stable levels within two months, confirming mean-reverting behavior in the reference period. The spread collapse around 2007 is an IDF weight compression artifact and is not used as a primary metric (see §6.5 and §5.4 for the embedding-based resolution).

In [ ]:
df_base_drift = compute_monthly_drift(df_baseline,   X_baseline,   centroid_baseline, 'baseline')
df_comp_drift = compute_monthly_drift(df_comparison, X_comparison, centroid_baseline, 'comparison')

print('=== Baseline Cosine Distance Statistics ===')
print(df_base_drift['cosine_distance_baseline'].describe().round(4).to_string())
print('\n=== Comparison Cosine Distance Statistics ===')
print(df_comp_drift['cosine_distance_baseline'].describe().round(4).to_string())

drift = df_comp_drift['cosine_distance_baseline'].mean() - df_base_drift['cosine_distance_baseline'].mean()
print(f'\nDrift (comparison mean - baseline mean): +{drift:.4f}')

In [ ]:
# Figure 1 — TF-IDF cosine drift over time
plot_drift(df_base_drift, df_comp_drift, str(FIG_DIR / 'fig1_tfidf_drift.png'))

In [ ]:
# Per-outlet cosine distance from baseline centroid
print('=== Per-Outlet Cosine Distance from Baseline Centroid ===')
for outlet in ALL_OUTLETS:
    idx = df_baseline[df_baseline['source'] == outlet].index.tolist()
    if len(idx) < 10:
        continue
    mat  = X_baseline[idx]
    ctrd = np.asarray(mat.mean(axis=0)).flatten()
    dist = cosine_distance(ctrd, centroid_baseline)
    print(f'  {outlet.upper():<10}: {dist:.4f}')

## 5. Permutation Test (§5.2, §4.6)

To confirm the observed drift reflects genuine temporal structure rather than random variation, a permutation test (n=1,000) is run. Month labels are randomly shuffled; the mean cosine distance from the baseline centroid is recomputed for each permutation. Under the null hypothesis of no temporal structure, permuted centroids converge toward the global comparison centroid, producing a tight distribution of low distances.

A p-value < 0.05 indicates the temporal ordering of headlines carries meaningful signal beyond random month assignment.

In [ ]:
# Note: this cell takes ~5–10 minutes to run
perm_results = permutation_test(
    df_baseline, df_comparison,
    X_baseline, X_comparison,
    centroid_baseline,
    n_permutations = 1000,
)

In [ ]:
# Figure 2 — Permutation test null distribution
plot_permutation_test(perm_results, str(FIG_DIR / 'fig2_permutation_test.png'))

## 6. Topic-Controlled Analysis (§5.3, §4.7)

To evaluate whether the observed drift reflects structural discourse change rather than simple topical vocabulary turnover, headlines containing keywords from three major discourse domains are removed iteratively and in combination. For each filtered subset, the TF-IDF vectorizer is refit on the filtered baseline and the centroid distance analysis is rerun.

Persistence of elevated cosine distance after filtering = evidence of structural rather than topical shift.

**Keyword sets (from §4.7):**
- COVID: covid, coronavirus, pandemic, vaccine, omicron, lockdown, fauci, pfizer, moderna, booster  
- Electoral: election, ballot, voter, voting, campaign, democrat, republican, senate, congress, midterm  
- Trump: trump, donald, maga, impeach

In [ ]:
filter_results = {}
print('=== Topic-Controlled Analysis ===')
print(f'\n{"Filter":<12} {"Comp Mean":>12} {"vs Unfiltered":>15} {"% Remaining":>13}')
print('-' * 55)
print(f'{"unfiltered":<12} {df_comp_drift["cosine_distance_baseline"].mean():>12.4f} {"—":>15} {"100.0%":>13}')

unfiltered_mean = df_comp_drift['cosine_distance_baseline'].mean()
for name, keywords in TOPIC_FILTERS.items():
    bs, cs = run_filtered_analysis(
        df_baseline, df_comparison,
        X_baseline, X_comparison,
        centroid_baseline, name, keywords,
    )
    filter_results[name] = (bs, cs)
    comp_mean = cs['cosine_distance_baseline'].mean()
    diff      = comp_mean - unfiltered_mean
    pct       = comp_mean / unfiltered_mean * 100
    print(f'{name:<12} {comp_mean:>12.4f} {diff:>+15.4f} {pct:>12.1f}%')

In [ ]:
# Figure 3 (paper) / Figure 5 (pipeline) — Topic-filtered drift
plot_topic_filters(df_comp_drift, filter_results, str(FIG_DIR / 'fig3_topic_filters.png'))

## 7. Sentence Embeddings (§5.4, §4.4)

Sentence embeddings using `all-MiniLM-L6-v2` provide a semantic signal beyond surface lexical patterns. Unlike TF-IDF, embeddings operate in a fixed, pre-trained 384-dimensional space independent of corpus vocabulary, producing more stable spread measurements.

Due to computational constraints, embeddings are constructed on a **stratified sample of 1,000 headlines per monthly bin per corpus**, preserving temporal and outlet distribution (§4.4). This reduces the encoding load to ~180,000 headlines.

> Both methods converging on sustained post-2016 upward drift, despite fundamentally different representational assumptions, strengthens the claim that the observed shifts reflect genuine structural change (§6.2).

In [ ]:
# Note: encoding takes ~20–30 minutes on CPU; use GPU if available
from sentence_transformers import SentenceTransformer

SAMPLE_PER_MONTH = 1000
rng = np.random.default_rng(42)

def stratified_sample(df, n_per_month, rng):
    """Sample up to n_per_month headlines from each monthly bin."""
    samples = []
    for month, group in df.groupby('year_month'):
        n = min(n_per_month, len(group))
        samples.append(group.sample(n, random_state=int(rng.integers(1e6))))
    return pd.concat(samples).reset_index(drop=True)

df_base_s = stratified_sample(df_baseline,   SAMPLE_PER_MONTH, rng)
df_comp_s = stratified_sample(df_comparison, SAMPLE_PER_MONTH, rng)

print(f'Stratified baseline   : {len(df_base_s):,} headlines')
print(f'Stratified comparison : {len(df_comp_s):,} headlines')

print('\nEncoding with all-MiniLM-L6-v2...')
model = SentenceTransformer('all-MiniLM-L6-v2')
base_embeddings = model.encode(df_base_s['title'].tolist(), show_progress_bar=True, batch_size=256)
comp_embeddings = model.encode(df_comp_s['title'].tolist(), show_progress_bar=True, batch_size=256)

print(f'\nBase embedding shape: {base_embeddings.shape}')
print(f'Comp embedding shape: {comp_embeddings.shape}')

In [ ]:
# Compute monthly drift in embedding space using same centroid distance approach
import scipy.sparse as sp

centroid_base_emb = base_embeddings.mean(axis=0)

def compute_monthly_drift_embeddings(df, embeddings, centroid_baseline_emb, corpus_label):
    """Monthly cosine drift for dense embedding representations."""
    records = []
    for month, group in df.groupby('year_month'):
        idx  = group.index.tolist()
        mat  = embeddings[idx]
        ctrd = mat.mean(axis=0)
        sim  = cosine_similarity(ctrd.reshape(1,-1), centroid_baseline_emb.reshape(1,-1))[0,0]
        dist = float(1.0 - sim)
        sims = cosine_similarity(mat, centroid_baseline_emb.reshape(1,-1)).flatten()
        records.append({
            'month':                    month,
            'cosine_distance_baseline': dist,
            'intra_month_spread':       float(np.std(sims)),
            'n_headlines':              len(idx),
            'corpus':                   corpus_label,
        })
    return pd.DataFrame(records).sort_values('month').reset_index(drop=True)

df_emb_base = compute_monthly_drift_embeddings(df_base_s, base_embeddings, centroid_base_emb, 'baseline')
df_emb_comp = compute_monthly_drift_embeddings(df_comp_s, comp_embeddings, centroid_base_emb, 'comparison')

print('=== Method Comparison: TF-IDF vs Sentence Embeddings ===')
print(f"{'Metric':<35} {'TF-IDF':>10} {'Embeddings':>12}")
print('-' * 60)
print(f"{'Baseline mean cosine dist':<35} {df_base_drift['cosine_distance_baseline'].mean():>10.4f} {df_emb_base['cosine_distance_baseline'].mean():>12.4f}")
print(f"{'Comparison mean cosine dist':<35} {df_comp_drift['cosine_distance_baseline'].mean():>10.4f} {df_emb_comp['cosine_distance_baseline'].mean():>12.4f}")
print(f"{'Drift (comp - base mean)':<35} {df_comp_drift['cosine_distance_baseline'].mean()-df_base_drift['cosine_distance_baseline'].mean():>+10.4f} {df_emb_comp['cosine_distance_baseline'].mean()-df_emb_base['cosine_distance_baseline'].mean():>+12.4f}")
print(f"{'Baseline spread mean':<35} {df_base_drift['intra_month_spread'].mean():>10.4f} {df_emb_base['intra_month_spread'].mean():>12.4f}")
print(f"{'Comparison spread mean':<35} {df_comp_drift['intra_month_spread'].mean():>10.4f} {df_emb_comp['intra_month_spread'].mean():>12.4f}")

In [ ]:
# Figure 4 — Sentence embedding drift
plot_drift(df_emb_base, df_emb_comp, str(FIG_DIR / 'fig4_embedding_drift.png'))

## 8. UMAP Visualization (§5.5, §4.9)

UMAP (Uniform Manifold Approximation and Projection) projects monthly centroid embeddings and individual headline embeddings onto 2D space, enabling visual inspection of the temporal discourse trajectory.

UMAP is selected over PCA for its ability to preserve local neighborhood structure in high-dimensional spaces (§4.9). Parameters: `n_neighbors=15`, `min_dist=0.1`, `metric=cosine`.

> **Interpretive note:** The UMAP visualization is illustrative rather than independently evidential — it cannot distinguish topical from structural change. Read alongside the topic-controlled cosine distance results (§6.3).

In [ ]:
import umap

# Compute monthly centroids in embedding space
all_months_sorted = sorted(set(df_base_s['year_month']) | set(df_comp_s['year_month']))
centroid_list, centroid_years, centroid_corpus = [], [], []

for ym in all_months_sorted:
    for corpus_label, df_s, embeddings in [('baseline', df_base_s, base_embeddings),
                                            ('comparison', df_comp_s, comp_embeddings)]:
        mask = df_s['year_month'] == ym
        if mask.sum() >= 20:
            ctrd = embeddings[df_s[mask].index.tolist()].mean(axis=0)
            centroid_list.append(ctrd)
            centroid_years.append(int(ym[:4]))
            centroid_corpus.append(corpus_label)

centroids_matrix = np.array(centroid_list)
print(f'Monthly centroids matrix: {centroids_matrix.shape}')

# Subsample individual headlines for Panel B (300 per corpus)
N_VIZ = 300
base_viz_idx = rng.choice(len(base_embeddings), N_VIZ, replace=False)
comp_viz_idx = rng.choice(len(comp_embeddings), N_VIZ, replace=False)
individual_matrix = np.vstack([base_embeddings[base_viz_idx], comp_embeddings[comp_viz_idx]])
print(f'Individual headline matrix: {individual_matrix.shape}')

In [ ]:
# Note: UMAP fit takes ~5 minutes
reducer_centroids = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
umap_centroids = reducer_centroids.fit_transform(centroids_matrix)

reducer_individual = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
umap_individual = reducer_individual.fit_transform(individual_matrix)

print(f'UMAP centroids shape   : {umap_centroids.shape}')
print(f'UMAP individual shape  : {umap_individual.shape}')

In [ ]:
# Figure 5 — UMAP projection (Panel A: centroids, Panel B: individual headlines)
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
    'Figure 5: UMAP Projection of Headline Embeddings (all-MiniLM-L6-v2)\n'
    'NYT · LAT · WSJ · Fox News · MSNBC  |  2001–2024',
    fontsize=13,
)

# Panel A — monthly centroids colored by year, temporal trajectory arrow
sc = axes[0].scatter(
    umap_centroids[:, 0], umap_centroids[:, 1],
    c=centroid_years, cmap='plasma', s=40, alpha=0.8,
)
plt.colorbar(sc, ax=axes[0], label='Year')
axes[0].set_title('A — Monthly Centroids Colored by Year', fontsize=11)
axes[0].set_xlabel('UMAP Dimension 1')
axes[0].set_ylabel('UMAP Dimension 2')

# Annotate key years
for label_year in [2001, 2008, 2016, 2020, 2024]:
    idx = [i for i, y in enumerate(centroid_years) if y == label_year]
    if idx:
        x, y = umap_centroids[idx[0], 0], umap_centroids[idx[0], 1]
        axes[0].annotate(str(label_year), (x, y), fontsize=8, fontweight='bold')

# Panel B — individual headlines by corpus
colors_b = ['steelblue'] * N_VIZ + ['darkorange'] * N_VIZ
axes[1].scatter(
    umap_individual[:N_VIZ, 0], umap_individual[:N_VIZ, 1],
    color='steelblue', s=15, alpha=0.5, label='Baseline (2001–2015)'
)
axes[1].scatter(
    umap_individual[N_VIZ:, 0], umap_individual[N_VIZ:, 1],
    color='darkorange', s=15, alpha=0.5, label='Comparison (2016–2024)'
)
axes[1].set_title('B — Individual Headlines: Baseline vs Comparison', fontsize=11)
axes[1].set_xlabel('UMAP Dimension 1')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(str(FIG_DIR / 'fig5_umap.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig5_umap.png')

## 9. Sentiment Analysis (§5.6, §4.8)

Headline sentiment is scored using VADER (Hutto & Gilbert, 2014), a rule-based model designed for short texts. Headlines are classified as:
- **Negative**: compound ≤ −0.05
- **Neutral**: −0.05 < compound < 0.05  
- **Positive**: compound ≥ 0.05

A **sentiment-conditioned cosine distance analysis** then partitions the corpus into three affective subsets and reruns the centroid distance analysis within each. The key finding: neutral headlines exhibit the *largest* drift (+0.172), suggesting the window shift is embedded in the background assumptions of factual reporting rather than driven by emotionally charged coverage (§6.4).

In [ ]:
# Score sentiment (takes a few minutes for 5.8M headlines)
sent_base_monthly, df_baseline = compute_sentiment(df_baseline, 'baseline')
sent_comp_monthly, df_comparison = compute_sentiment(df_comparison, 'comparison')

# Aggregate sentiment statistics (Table 1 in paper)
for corpus_label, df in [('Baseline', df_baseline), ('Comparison', df_comparison)]:
    neg = (df['sentiment_bin'] == 'negative').mean() * 100
    neu = (df['sentiment_bin'] == 'neutral').mean()  * 100
    pos = (df['sentiment_bin'] == 'positive').mean() * 100
    print(f'{corpus_label}: neg={neg:.1f}%  neu={neu:.1f}%  pos={pos:.1f}%')

In [ ]:
# Statistical test: are the two sentiment distributions significantly different?
b_sent = df_baseline['compound'].values
c_sent = df_comparison['compound'].values
t_stat, p_val = stats.ttest_ind(b_sent, c_sent)
cohen_d = (c_sent.mean() - b_sent.mean()) / \
          np.sqrt((b_sent.std()**2 + c_sent.std()**2) / 2)

print(f'Baseline mean sentiment  : {b_sent.mean():.4f} (std={b_sent.std():.4f})')
print(f'Comparison mean sentiment: {c_sent.mean():.4f} (std={c_sent.std():.4f})')
print(f't-statistic : {t_stat:.4f}')
print(f'p-value     : {p_val:.4f}')
print(f"Cohen's d   : {cohen_d:.4f}")

In [ ]:
# Figure 6 — Three-panel sentiment analysis
plot_sentiment(sent_base_monthly, sent_comp_monthly, str(FIG_DIR / 'fig6_sentiment.png'))

In [ ]:
# Sentiment-conditioned drift (Table 4 in paper)
results_by_sentiment = sentiment_conditioned_drift(
    df_baseline, df_comparison,
    X_baseline, X_comparison,
    centroid_baseline,
)

print('=== Sentiment-Conditioned Cosine Distance (Table 4) ===')
print(f"{'Sentiment Bin':<15} {'Baseline Mean':>15} {'Comparison Mean':>17} {'Drift':>8}")
print('-' * 58)
for sent_bin in ['negative', 'neutral', 'positive']:
    if sent_bin not in results_by_sentiment:
        continue
    df_s = results_by_sentiment[sent_bin]
    base_mean = df_s[df_s['corpus'] == 'baseline']['cosine_dist'].mean()
    comp_mean = df_s[df_s['corpus'] == 'comparison']['cosine_dist'].mean()
    drift = comp_mean - base_mean
    print(f'{sent_bin:<15} {base_mean:>15.4f} {comp_mean:>17.4f} {drift:>+8.4f}')

In [ ]:
# Figure 7 — Sentiment-conditioned drift
plot_sentiment_conditioned_drift(
    results_by_sentiment, sent_base_monthly,
    str(FIG_DIR / 'fig7_sentiment_conditioned.png')
)

## 10. Save Results

In [ ]:
save_results(
    df_base_drift, df_comp_drift,
    sent_base_monthly, sent_comp_monthly,
    str(RES_DIR),
)

# Also save embedding-based monthly stats
df_emb_base.to_csv(RES_DIR / 'baseline_monthly_drift_embeddings.csv', index=False)
df_emb_comp.to_csv(RES_DIR / 'comparison_monthly_drift_embeddings.csv', index=False)
print('Embedding stats saved.')

print('\n✓ All outputs saved.')
print(f'  Figures : {FIG_DIR}/')
print(f'  Results : {RES_DIR}/')

---

## References

- Baly et al. (2020). We Can Detect Your Bias. *EMNLP 2020.*
- Card et al. (2015). The Media Frames Corpus. *ACL 2015.*
- Chair for Data Science, University of Mannheim (2026). US Multi-Outlet News Headlines 2001–2024. HuggingFace.
- Gentzkow, Shapiro & Taddy (2019). Measuring Group Differences in High-Dimensional Choices. *Econometrica.*
- Hamilton, Leskovec & Jurafsky (2016). Diachronic Word Embeddings. *ACL 2016.*
- Hutto & Gilbert (2014). VADER. *ICWSM 2014.*
- Leeb & Schölkopf (2024). Babel Briefings. *NAACL 2024.*
- Vosoughi, Roy & Aral (2018). The Spread of True and False News Online. *Science.*